In [1]:
import numpy as np, chromadb
from chromadb.config import Settings
from langchain_ollama import OllamaEmbeddings

emb = OllamaEmbeddings(model="nomic-embed-text")
sentences = [
    "How do I control weeds in sesame?",
    "Which herbicide works against weeds in a sesame field?",
    "Sesame leaf spot disease treatment",
    "Best time to sow wheat in Punjab",
    "The India versus Australia cricket match starts at 2 pm",
    "How to fix a segmentation fault in C++",
]
E = np.array(emb.embed_documents(sentences))
q = np.array(emb.embed_query("How do I control weeds in sesame?"))
print("vector lengths:", np.round(np.linalg.norm(E, axis=1), 3))

vector lengths: [1. 1. 1. 1. 1. 1.]


In [2]:
client = chromadb.PersistentClient(path="../data/chroma/toy",
                                   settings=Settings(anonymized_telemetry=False))
ids = [str(i) for i in range(len(sentences))]

col_cos = client.get_or_create_collection(
    "toy_cosine", configuration={"hnsw": {"space": "cosine"}})
col_def = client.get_or_create_collection("toy_default")
for col in (col_cos, col_def):
    col.add(ids=ids, embeddings=E.tolist(), documents=sentences)

def cos(a, b): return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

for name, col in (("cosine", col_cos), ("default", col_def)):
    res = col.query(query_embeddings=[q.tolist()], n_results=6)
    print(name)
    for i, dist in zip(res["ids"][0], res["distances"][0]):
        i = int(i)
        print(round(dist, 4),
              "| 1-cos:", round(1 - cos(q, E[i]), 4),
              "| sqL2:", round(float(np.sum((q - E[i]) ** 2)), 4),
              "|", sentences[i][:40])

cosine
0.0 | 1-cos: -0.0 | sqL2: 0.0 | How do I control weeds in sesame?
0.1332 | 1-cos: 0.1332 | sqL2: 0.2664 | Which herbicide works against weeds in a
0.2825 | 1-cos: 0.2825 | sqL2: 0.5651 | Sesame leaf spot disease treatment
0.4976 | 1-cos: 0.4976 | sqL2: 0.9952 | Best time to sow wheat in Punjab
0.6052 | 1-cos: 0.6052 | sqL2: 1.2104 | How to fix a segmentation fault in C++
0.6422 | 1-cos: 0.6422 | sqL2: 1.2845 | The India versus Australia cricket match
default
0.0 | 1-cos: -0.0 | sqL2: 0.0 | How do I control weeds in sesame?
0.2664 | 1-cos: 0.1332 | sqL2: 0.2664 | Which herbicide works against weeds in a
0.5651 | 1-cos: 0.2825 | sqL2: 0.5651 | Sesame leaf spot disease treatment
0.9952 | 1-cos: 0.4976 | sqL2: 0.9952 | Best time to sow wheat in Punjab
1.2104 | 1-cos: 0.6052 | sqL2: 1.2104 | How to fix a segmentation fault in C++
1.2845 | 1-cos: 0.6422 | sqL2: 1.2845 | The India versus Australia cricket match


In [1]:
from importlib.metadata import version

for pkg in ["chromadb", "langchain-core", "langchain-ollama", "langchain-text-splitters", "pymupdf"]:
    print(f"{pkg}: {version(pkg)}")

chromadb: 1.5.9
langchain-core: 1.6.10
langchain-ollama: 1.1.0
langchain-text-splitters: 1.1.3
pymupdf: 1.28.2


In [2]:
import csv
from pathlib import Path

def load_questions(csv_path="../eval/questions.csv"):
    rows = []
    with open(csv_path, mode="r", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            pages = [int(p.strip()) for p in r["expected_pages"].split(";") if p.strip()]
            phrases = [p.strip().lower() for p in r["answer_phrases"].split(";") if p.strip()]
            rows.append({
                "id": r["id"],
                "question": r["question"],
                "type": r["type"],
                "tag": r["tag"],
                "doc_id": r["doc_id"].strip(),
                "expected_pages": pages,
                "answer_phrases": phrases,
                "match": r["match"].strip(),
                "notes": r["notes"].strip(),
            })
    return rows

questions = load_questions()
print(f"Loaded {len(questions)} questions.")
for q in questions:
    print(q["id"], q["type"], q["tag"], q["expected_pages"], q["answer_phrases"])

Loaded 10 questions.
q1 in_scope table [20] ['2/100 leaves']
q2 in_scope multi [29] ['3 to 26', '6 to 15']
q3 in_scope plain [34] ['clear water']
q4 in_scope conflict [4, 7] ['25-28 dae', '18-20 das']
q5 in_scope plain [3] ['2% brine']
q6 in_scope plain [5] ['clearfield']
q7 in_scope table [5] ['120:60:40']
q8 in_scope paraphrase [56] ['solarization']
q9 out_of_scope out_of_scope [] []
q10 out_of_scope out_of_scope [] []


In [3]:
import chromadb
from chromadb.config import Settings
from langchain_ollama import OllamaEmbeddings

EMBED_MODEL = "nomic-embed-text"
EXPECTED_DIM = 768
EXPECTED_PREFIX = "none"

emb = OllamaEmbeddings(model=EMBED_MODEL)
client = chromadb.PersistentClient(
    path="../data/chroma/mitra",
    settings=Settings(anonymized_telemetry=False)
)
col = client.get_collection("mitra_chunks")

# Guard: stop immediately if collection metadata does not match query settings
meta = col.metadata or {}
assert meta.get("embed_model") == EMBED_MODEL, f"Model mismatch: {meta}"
assert meta.get("dim") == EXPECTED_DIM, f"Dim mismatch: {meta}"
assert meta.get("doc_prefix") == EXPECTED_PREFIX, f"Prefix mismatch: {meta}"

def norm_text(t):
    return " ".join(t.lower().splitlines())

def chunk_matches_phrase(chunk, q, phrase):
    if chunk["doc_id"] != q["doc_id"]:
        return False
    if chunk["page"] not in q["expected_pages"]:
        return False
    return phrase in norm_text(chunk["text"])

def eval_hits_at_k(chunks_k, q):
    if q["match"] == "any":
        return any(
            chunk_matches_phrase(c, q, ph)
            for c in chunks_k
            for ph in q["answer_phrases"]
        )
    elif q["match"] == "all":
        return all(
            any(chunk_matches_phrase(c, q, ph) for c in chunks_k)
            for ph in q["answer_phrases"]
        )
    raise ValueError(f"Unknown match rule: {q['match']}")

def first_hit_rank(chunks, q):
    for r in range(1, len(chunks) + 1):
        if eval_hits_at_k(chunks[:r], q):
            return r
    return None

eval_records = []
misses = []

for q in questions:
    qv = emb.embed_query(q["question"])
    res = col.query(query_embeddings=[qv], n_results=5)
    
    top5 = []
    for cid, dist, m, doc in zip(
        res["ids"][0], res["distances"][0], res["metadatas"][0], res["documents"][0]
    ):
        top5.append({
            "id": cid,
            "doc_id": m["doc_id"],
            "page": int(m["page"]),
            "similarity": 1.0 - float(dist),
            "text": doc,
        })
    
    if q["type"] == "in_scope":
        h1 = eval_hits_at_k(top5[:1], q)
        h3 = eval_hits_at_k(top5[:3], q)
        h5 = eval_hits_at_k(top5[:5], q)
        rank = first_hit_rank(top5, q)
        eval_records.append({
            "id": q["id"],
            "type": q["type"],
            "tag": q["tag"],
            "hit1": h1,
            "hit3": h3,
            "hit5": h5,
            "first_rank": rank,
            "top1_sim": round(top5[0]["similarity"], 4),
            "top5": top5,
        })
        if not h5:
            misses.append((q, top5[:3]))
    else:
        eval_records.append({
            "id": q["id"],
            "type": q["type"],
            "tag": q["tag"],
            "hit1": None,
            "hit3": None,
            "hit5": None,
            "first_rank": None,
            "top1_sim": round(top5[0]["similarity"], 4),
            "top5": top5,
        })

print("ID  | Tag        | Hit@1 | Hit@3 | Hit@5 | FirstRank | Top1_Sim")
print("-" * 65)
for r in eval_records:
    if r["type"] == "in_scope":
        print(f"{r['id']:<3} | {r['tag']:<10} | {str(r['hit1']):<5} | {str(r['hit3']):<5} | {str(r['hit5']):<5} | {str(r['first_rank']):<9} | {r['top1_sim']:.4f}")
    else:
        print(f"{r['id']:<3} | {r['tag']:<10} |  -    |  -    |  -    |  -        | {r['top1_sim']:.4f}")

ID  | Tag        | Hit@1 | Hit@3 | Hit@5 | FirstRank | Top1_Sim
-----------------------------------------------------------------
q1  | table      | False | False | False | None      | 0.8901
q2  | multi      | True  | True  | True  | 1         | 0.8391
q3  | plain      | True  | True  | True  | 1         | 0.8141
q4  | conflict   | False | False | False | None      | 0.7722
q5  | plain      | False | True  | True  | 3         | 0.8067
q6  | plain      | True  | True  | True  | 1         | 0.8585
q7  | table      | False | True  | True  | 2         | 0.7776
q8  | paraphrase | False | True  | True  | 3         | 0.7743
q9  | out_of_scope |  -    |  -    |  -    |  -        | 0.7557
q10 | out_of_scope |  -    |  -    |  -    |  -        | 0.6249


In [4]:
for q, r in zip(questions, eval_records):
    if q["type"] == "in_scope" and not r["hit3"]:
        print("=" * 70)
        print(f"MISS (or outside top-3): {q['id']} [{q['tag']}] -> {q['question']}")
        print(f"Expected: {q['doc_id']} pages={q['expected_pages']} phrases={q['answer_phrases']}")
        for idx, c in enumerate(r["top5"][:3], start=1):
            clean_preview = norm_text(c["text"])[:300]
            print(f"  [{idx}] sim={c['similarity']:.4f} | {c['doc_id']} p.{c['page']}")
            print(f"      {clean_preview}")

MISS (or outside top-3): q1 [table] -> What is the economic threshold level (ETL) for aphids and whiteflies when monitoring potato pests?
Expected: potato_disease-pest-management_cpri_2016_manual pages=[20] phrases=['2/100 leaves']
  [1] sim=0.8901 | potato_disease-pest-management_cpri_2016_manual p.19
      prevent the pests from causing unacceptable injury or harm. a threshold often is set at the level where the economic losses caused by pest damage, if the pest population continued to grow, would be greater than the cost of controlling the pests. these types of action thresholds sometimes are called 
  [2] sim=0.7718 | potato_disease-pest-management_cpri_2016_manual p.19
      • are the numbers great enough to warrant control? • when is the right time to begin control? • have the control efforts successfully reduced the number of pests? monitoring of arthropod pests is done by trapping or by scouting. monitoring of weeds is usually done by visual inspection. monitoring fo
  [3] sim=

In [5]:
import numpy as np

# Fetch all stored IDs and embeddings from Chroma
stored = col.get(include=["embeddings"])
all_ids = stored["ids"]
all_embs = np.array(stored["embeddings"], dtype=np.float64)

# Normalize rows to unit length for exact cosine via dot product
all_embs_norm = all_embs / np.linalg.norm(all_embs, axis=1, keepdims=True)
print("Fetched stored matrix:", all_embs_norm.shape)

bf_comparison = []
for q, r in zip(questions, eval_records):
    qv = np.array(emb.embed_query(q["question"]), dtype=np.float64)
    qv_norm = qv / np.linalg.norm(qv)
    
    # Exact brute-force cosine scores against all 615 chunks
    sims = all_embs_norm @ qv_norm
    bf_top5_idx = np.argsort(-sims)[:5]
    bf_top5_ids = [all_ids[i] for i in bf_top5_idx]
    
    chroma_top5_ids = [c["id"] for c in r["top5"]]
    
    overlap_at_5 = len(set(chroma_top5_ids) & set(bf_top5_ids))
    top1_same = (chroma_top5_ids[0] == bf_top5_ids[0])
    
    bf_comparison.append({
        "id": q["id"],
        "overlap_at_5": overlap_at_5,
        "top1_same": top1_same,
        "chroma_top1": chroma_top5_ids[0],
        "bf_top1": bf_top5_ids[0],
    })
    print(f"{q['id']:<3} | overlap@5: {overlap_at_5}/5 | top1_same: {str(top1_same):<5} | top1: {chroma_top5_ids[0]}")

Fetched stored matrix: (615, 768)
q1  | overlap@5: 5/5 | top1_same: True  | top1: potato_disease-pest-management_cpri_2016_manual:p19:c2
q2  | overlap@5: 5/5 | top1_same: True  | top1: potato_disease-pest-management_cpri_2016_manual:p29:c1
q3  | overlap@5: 5/5 | top1_same: True  | top1: potato_disease-pest-management_cpri_2016_manual:p34:c2
q4  | overlap@5: 5/5 | top1_same: True  | top1: rice_weed-mgmt-dry-direct-seeded_nrri_2019_brief:p6:c0
q5  | overlap@5: 5/5 | top1_same: True  | top1: rice_weedy-rice-options_dwsr_2013_article:p4:c0
q6  | overlap@5: 5/5 | top1_same: True  | top1: rice_weedy-rice-options_dwsr_2013_article:p5:c1
q7  | overlap@5: 5/5 | top1_same: True  | top1: wheat_rust-resistant-varieties-economics_vpkas_2020_article:p4:c5
q8  | overlap@5: 5/5 | top1_same: True  | top1: potato_disease-pest-management_cpri_2016_manual:p123:c1
q9  | overlap@5: 5/5 | top1_same: True  | top1: rice_weed-mgmt-dry-direct-seeded_nrri_2019_brief:p7:c0
q10 | overlap@5: 5/5 | top1_same: True  |

In [6]:
in_scores = [(r["id"], r["tag"], r["top1_sim"], r["hit3"], r["hit5"]) for r in eval_records if r["type"] == "in_scope"]
out_scores = [(r["id"], r["tag"], r["top1_sim"]) for r in eval_records if r["type"] == "out_of_scope"]

print("=== IN-SCOPE TOP-1 SIMILARITIES ===")
for qid, tag, sim, h3, h5 in sorted(in_scores, key=lambda x: -x[2]):
    print(f"  {qid:<3} ({tag:<10}): top1_sim={sim:.4f} | hit@3={h3} | hit@5={h5}")

print("\n=== OUT-OF-SCOPE TOP-1 SIMILARITIES ===")
for qid, tag, sim in sorted(out_scores, key=lambda x: -x[2]):
    print(f"  {qid:<3} ({tag:<10}): top1_sim={sim:.4f}")

min_in = min(x[2] for x in in_scores)
max_out = max(x[2] for x in out_scores)
print(f"\nMin in-scope top1_sim:     {min_in:.4f}")
print(f"Max out-of-scope top1_sim: {max_out:.4f}")
print(f"Separation gap (min_in - max_out): {min_in - max_out:.4f}")

# Save summary to ../eval/stage3_results.md
lines = [
    "# Stage 3 Retrieval Evaluation Results\n",
    "## 1. Hit@k Table (In-Scope & Out-of-Scope)\n",
    "| ID | Type | Tag | Hit@1 | Hit@3 | Hit@5 | First Rank | Top-1 Similarity |",
    "|---|---|---|---|---|---|---|---|",
]
for r in eval_records:
    lines.append(
        f"| {r['id']} | {r['type']} | {r['tag']} | {r['hit1']} | {r['hit3']} | {r['hit5']} | {r['first_rank']} | {r['top1_sim']:.4f} |"
    )

lines.append("\n## 2. Chroma HNSW vs. Brute-Force NumPy Top-5\n")
lines.append("| ID | Overlap@5 | Top-1 Same | Chroma Top-1 ID | Brute-Force Top-1 ID |")
lines.append("|---|---|---|---|---|")
for b in bf_comparison:
    lines.append(
        f"| {b['id']} | {b['overlap_at_5']}/5 | {b['top1_same']} | `{b['chroma_top1']}` | `{b['bf_top1']}` |"
    )

lines.append("\n## 3. Score Distribution Summary\n")
lines.append(f"- **Min in-scope top-1 similarity:** `{min_in:.4f}`")
lines.append(f"- **Max out-of-scope top-1 similarity:** `{max_out:.4f}`")
lines.append(f"- **Gap (`min_in - max_out`):** `{min_in - max_out:.4f}`\n")

Path("../eval/stage3_results.md").write_text("\n".join(lines), encoding="utf-8")
print("Saved ../eval/stage3_results.md")

=== IN-SCOPE TOP-1 SIMILARITIES ===
  q1  (table     ): top1_sim=0.8901 | hit@3=False | hit@5=False
  q6  (plain     ): top1_sim=0.8585 | hit@3=True | hit@5=True
  q2  (multi     ): top1_sim=0.8391 | hit@3=True | hit@5=True
  q3  (plain     ): top1_sim=0.8141 | hit@3=True | hit@5=True
  q5  (plain     ): top1_sim=0.8067 | hit@3=True | hit@5=True
  q7  (table     ): top1_sim=0.7776 | hit@3=True | hit@5=True
  q8  (paraphrase): top1_sim=0.7743 | hit@3=True | hit@5=True
  q4  (conflict  ): top1_sim=0.7722 | hit@3=False | hit@5=False

=== OUT-OF-SCOPE TOP-1 SIMILARITIES ===
  q9  (out_of_scope): top1_sim=0.7557
  q10 (out_of_scope): top1_sim=0.6249

Min in-scope top1_sim:     0.7722
Max out-of-scope top1_sim: 0.7557
Separation gap (min_in - max_out): 0.0165
Saved ../eval/stage3_results.md
